# landing_engine\nGenerated from 02_processing/landing_engine.py for Databricks notebook import.\n

In [ ]:
"""Landing engine: preserve source records and add technical metadata."""\n\nfrom __future__ import annotations\n\n\ndef add_landing_metadata(df, source_system: str, source_entity: str, load_id: str):\n    from pyspark.sql import functions as F\n\n    return (\n        df.withColumn("source_system", F.lit(source_system))\n        .withColumn("source_entity", F.lit(source_entity))\n        .withColumn("load_id", F.lit(load_id))\n        .withColumn("ingest_ts", F.current_timestamp())\n    )\n\n\nfrom typing import Optional\ndef write_landing(\n    df,\n    table_name: str,\n    archive_table: Optional[str] = None,\n    retention_days: Optional[int] = None\n):\n    """\n    Write DataFrame to landing table and optionally to an archive table with retention policy.\n    Args:\n        df: DataFrame to write\n        table_name: Main landing table name\n        archive_table: Optional archive table name\n        retention_days: Optional retention period in days for archive\n    """\n    if not table_name or not table_name.strip():\n        raise ValueError("landing table_name must not be empty")\n    (\n        df.write.mode("append")\n        .format("delta")\n        .option("mergeSchema", "true")\n        .saveAsTable(table_name)\n    )\n    if archive_table and archive_table.strip():\n        (\n            df.write.mode("append")\n            .format("delta")\n            .option("mergeSchema", "true")\n            .saveAsTable(archive_table)\n        )\n        # Optionally enforce retention by vacuuming old data\n        if retention_days is not None:\n            from pyspark.sql import SparkSession\n            spark = SparkSession.getActiveSession()\n            if spark:\n                spark.sql(f"VACUUM {archive_table} RETAIN {retention_days} HOURS")\n